# Training with DPSGD

In order to implement DPSGD trining you need to implement DPSGD optimizer, which clips and adds noise to the gradients.

### Modified code:
- crypten/optim/__init__.py to export also DPSGD class
- Mnist_utils modified to save directly the one hot encoded version of the labels
- crypten/__init__.py modified load from party to enable the requires_grad (autograd) functionality for labls loaded directly from file

## Known issues: 
- Need to set the batch size to one to have per example gradient clipping
- Need to set properly the noise, the Poisson lambda for Skellam depends on the quantization
- The quantization is not implemented, currently it is fixed point

In [1]:
import crypten
import torch


crypten.init()
torch.set_num_threads(1)

In [9]:
# Script used to split the dataset, horizontally, vertically, between features and labels

%run ./mnist_utils.py --option data --reduced 100 --binary

In [4]:
import torch.nn as nn
import torch.nn.functional as F

#Define an example network
class ExampleNet(nn.Module):
    def __init__(self):
        super(ExampleNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=0)
        self.fc1 = nn.Linear(16 * 12 * 12, 100)
        self.fc2 = nn.Linear(100, 2) # For binary classification, final layer needs only 2 outputs
 
    def forward(self, x):
        out = self.conv1(x)
        out = F.relu(out)
        out = F.max_pool2d(out, 2)
        out = out.view(-1, 16 * 12 * 12)
        out = self.fc1(out)
        out = F.relu(out)
        out = self.fc2(out)
        return out
    
crypten.common.serial.register_safe_class(ExampleNet)

In [5]:
# Define source argument values for Alice and Bob
ALICE = 0
BOB = 1

In [6]:
# Load Alice's data 
data_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE)

In [7]:
# We'll now set up the data for our small example below
# For illustration purposes, we will create toy data
# and encrypt all of it from source ALICE
x_small = torch.rand(100, 1, 28, 28)
y_small = torch.randint(1, (100,))

# Transform labels into one-hot encoding
label_eye = torch.eye(2)
y_one_hot = label_eye[y_small]

# Transform all data to CrypTensors
x_train = crypten.cryptensor(x_small, src=ALICE)
y_train = crypten.cryptensor(y_one_hot)


# Instantiate and encrypt a CrypTen model
model_plaintext = ExampleNet()
dummy_input = torch.empty(1, 1, 28, 28)
model = crypten.nn.from_pytorch(model_plaintext, dummy_input)
model.encrypt()

/usr/local/lib/python3.7/site-packages/crypten-0.4.0-py3.7.egg/crypten/nn/onnx_converter.py:176: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /root/pytorch/torch/csrc/utils/tensor_numpy.cpp:199.)
  param = torch.from_numpy(numpy_helper.to_array(node))


Graph encrypted module

# 1 Party example

In [7]:
# Example: Stochastic Gradient Descent in CrypTen

model.train() # Change to training mode
loss = crypten.nn.MSELoss() # Choose loss functions

# Set parameters: learning rate, num_epochs
learning_rate = 0.001
num_epochs = 2
optimizer = crypten.optim.DPSGD(model.parameters(), learning_rate)
# Train the model: SGD on encrypted data
for i in range(num_epochs):

    # forward pass
    output = model(x_train)
    loss_value = loss(output, y_train)
    
    
    # set gradients to zero
    model.zero_grad()

    # perform backward pass
    loss_value.backward()

    optimizer.step()
    # update parameters
    #model.update_parameters(learning_rate) 
    
    # examine the loss after each epoch
    print("Epoch: {0:d} Loss: {1:.4f}".format(i, loss_value.get_plain_text()))



Epoch: 0 Loss: 0.4167
Epoch: 1 Loss: 0.3970


# 2pc complete example

In [15]:
import crypten.mpc as mpc
import crypten.communicator as comm

# Convert labels to one-hot encoding
# Since labels are public in this use case, we will simply use them from loaded torch tensors
#labels = torch.load('/tmp/train_labels.pth')
#labels = labels.long()
#labels_one_hot = label_eye[labels]

@mpc.run_multiprocess(world_size=2)
def run_encrypted_training():
    # Load data:
    x_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE)
    x_bob_enc = crypten.load_from_party('/tmp/bob_train.pth', src=BOB)

    # Labels already save as one-hot
    y_alice_label_enc = crypten.load_from_party('/tmp/alice_train_labels_onehot.pth', src=ALICE, requires_grad=True)    
    y_bob_label_enc = crypten.load_from_party('/tmp/bob_train_labels_onehot.pth', src=BOB,requires_grad=True)    
    #crypten.print(x_alice_enc.size())
    #crypten.print(x_bob_enc.size())
    
    # Combine the feature sets: identical to Tutorial 3
    x_combined_enc = crypten.cat([x_alice_enc, x_bob_enc], dim=0)
    # Reshape to match the network architecture
    x_combined_enc = x_combined_enc.unsqueeze(1)
    
    y_combined_enc_one_hot = crypten.cat([y_alice_label_enc,y_bob_label_enc], dim=0)

    #crypten.print(f"Combined labels: {y_combined_enc_one_hot.get_plain_text()}")
    # Initialize a plaintext model and convert to CrypTen model
    pytorch_model = ExampleNet()
    model = crypten.nn.from_pytorch(pytorch_model, dummy_input)
    model.encrypt()
    # Set train mode
    model.train()
  
    # Define a loss function
    loss = crypten.nn.MSELoss()

    # Define training parameters
    learning_rate = 0.001
    num_epochs = 2
    batch_size = 1
    num_batches = x_combined_enc.size(0) // batch_size
    optimizer = crypten.optim.DPSGD(model.parameters(), learning_rate)
    
    rank = comm.get().get_rank()
    for i in range(num_epochs): 
        crypten.print(f"Epoch {i} in progress:")       
        
        for batch in range(num_batches):
            # define the start and end of the training mini-batch
            start, end = batch * batch_size, (batch + 1) * batch_size
                                    
            # construct CrypTensors out of training examples / labels
            x_train = x_combined_enc[start:end]
            y_train = y_combined_enc_one_hot[start:end]
            #crypten.print(f"y batch type: {type(y_batch)}", in_order=True)
            #crypten.print(f"y grad batch type : {type(y_batch.grad)}", in_order=True)
            #y_train = y_batch#.grad #crypten.cryptensor(y_batch, requires_grad=True)
            
            # perform forward pass:
            output = model(x_train)
            loss_value = loss(output, y_train)
            # set gradients to "zero" 
            model.zero_grad()

            # perform backward pass: 
            loss_value.backward()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value backward: {model.}", in_order=True)
            
            # update parameters
            optimizer.step()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tModel: {model}", in_order=True)

            
            # Print progress every batch:
            batch_loss = loss_value.get_plain_text()
            crypten.print(f"\tBatch {(batch + 1)} of {num_batches} Loss {batch_loss.item():.4f}")

run_encrypted_training()

Epoch 0 in progress:
	Batch 1 of 100 Loss 0.5475
	Batch 2 of 100 Loss 0.5893
	Batch 3 of 100 Loss 0.6514
	Batch 4 of 100 Loss 0.6487
	Batch 5 of 100 Loss 0.5296
	Batch 6 of 100 Loss 0.5369
	Batch 7 of 100 Loss 0.5465
	Batch 8 of 100 Loss 0.5039
	Batch 9 of 100 Loss 0.4396
	Batch 10 of 100 Loss 0.3244
	Batch 11 of 100 Loss 0.3258
	Batch 12 of 100 Loss 0.4780
	Batch 13 of 100 Loss 0.2804
	Batch 14 of 100 Loss 0.3987
	Batch 15 of 100 Loss 0.3454
	Batch 16 of 100 Loss 0.2828
	Batch 17 of 100 Loss 0.2560
	Batch 18 of 100 Loss 0.1775
	Batch 19 of 100 Loss 0.3081
	Batch 20 of 100 Loss 0.1573
	Batch 21 of 100 Loss 0.2251
	Batch 22 of 100 Loss 0.5881
	Batch 23 of 100 Loss 0.1940
	Batch 24 of 100 Loss 0.2551
	Batch 25 of 100 Loss 0.1033
	Batch 26 of 100 Loss 0.1021
	Batch 27 of 100 Loss 0.1446
	Batch 28 of 100 Loss 0.0174
	Batch 29 of 100 Loss 0.0945
	Batch 30 of 100 Loss 0.1521
	Batch 31 of 100 Loss 0.0710
	Batch 32 of 100 Loss 0.0162
	Batch 33 of 100 Loss 0.1558
	Batch 34 of 100 Loss 0.0571
	B

[None, None]

## Cleartext


In [13]:

# Convert labels to one-hot encoding
# Since labels are public in this use case, we will simply use them from loaded torch tensors
#labels = torch.load('/tmp/train_labels.pth')
#labels = labels.long()
#labels_one_hot = label_eye[labels]

def run_clear_training():
    x_alice = torch.load('/tmp/alice_train.pth')
    x_bob = torch.load('/tmp/bob_train.pth')

    y_alice = torch.load('/tmp/alice_train_labels.pth')
    y_bob = torch.load('/tmp/bob_train_labels.pth')

    x_combined = torch.cat([x_alice, x_bob]).unsqueeze(1)
    y_combined = torch.cat([y_alice, y_bob])

    model = ExampleNet()
    
    # Set train mode
    model.train()
  
    # Define a loss function
    loss = torch.nn.MSELoss()

    # Define training parameters
    learning_rate = 0.001
    num_epochs = 2
    batch_size = 1
    num_batches = x_combined.size(0) // batch_size
    optimizer = torch.optim.SGD(model.parameters(), learning_rate)
    
    for i in range(num_epochs): 
        print(f"Epoch {i} in progress:")       
        
        for batch in range(num_batches):
            # define the start and end of the training mini-batch
            start, end = batch * batch_size, (batch + 1) * batch_size
                                    
            # construct CrypTensors out of training examples / labels
            x_train = x_combined[start:end]
            y_train = y_combined[start:end]
            #crypten.print(f"y batch type: {type(y_batch)}", in_order=True)
            #crypten.print(f"y grad batch type : {type(y_batch.grad)}", in_order=True)
            #y_train = y_batch#.grad #crypten.cryptensor(y_batch, requires_grad=True)
            
            # perform forward pass:
            output = model(x_train)
            loss_value = loss(output, y_train)
            # set gradients to "zero" 
            model.zero_grad()

            # perform backward pass: 
            loss_value.backward()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value backward: {model.}", in_order=True)
            
            # update parameters
            optimizer.step()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tModel: {model}", in_order=True)

            
            # Print progress every batch:
            batch_loss = loss_value
            print(f"\tBatch {(batch + 1)} of {num_batches} Loss {batch_loss.item():.4f}")

run_clear_training()

Epoch 0 in progress:
	Batch 1 of 100 Loss 1.3497
	Batch 2 of 100 Loss 0.0249
	Batch 3 of 100 Loss 1.1652
	Batch 4 of 100 Loss 1.2110
	Batch 5 of 100 Loss 0.9897
	Batch 6 of 100 Loss 0.6669
	Batch 7 of 100 Loss 0.8188
	Batch 8 of 100 Loss 0.3050
	Batch 9 of 100 Loss 0.7192
	Batch 10 of 100 Loss 0.4177
	Batch 11 of 100 Loss 0.2727
	Batch 12 of 100 Loss 0.4943
	Batch 13 of 100 Loss 0.1997
	Batch 14 of 100 Loss 0.2857
	Batch 15 of 100 Loss 0.3781
	Batch 16 of 100 Loss 0.1436
	Batch 17 of 100 Loss 0.1397
	Batch 18 of 100 Loss 0.0115
	Batch 19 of 100 Loss 0.2931
	Batch 20 of 100 Loss 0.0809
	Batch 21 of 100 Loss 0.0888
	Batch 22 of 100 Loss 0.6007
	Batch 23 of 100 Loss 0.1366
	Batch 24 of 100 Loss 0.1785
	Batch 25 of 100 Loss 0.1838
	Batch 26 of 100 Loss 0.0442
	Batch 27 of 100 Loss 0.2384
	Batch 28 of 100 Loss 0.0878
	Batch 29 of 100 Loss 0.0220
	Batch 30 of 100 Loss 0.0929
	Batch 31 of 100 Loss 0.0595
	Batch 32 of 100 Loss 0.0151
	Batch 33 of 100 Loss 0.1366
	Batch 34 of 100 Loss 0.0177
	B

# Without DP

In [12]:
import crypten.mpc as mpc
import crypten.communicator as comm

# Convert labels to one-hot encoding
# Since labels are public in this use case, we will simply use them from loaded torch tensors
#labels = torch.load('/tmp/train_labels.pth')
#labels = labels.long()
#labels_one_hot = label_eye[labels]

@mpc.run_multiprocess(world_size=2)
def run_encrypted_training():
    # Load data:
    x_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE)
    x_bob_enc = crypten.load_from_party('/tmp/bob_train.pth', src=BOB)

    # Labels already save as one-hot
    y_alice_label_enc = crypten.load_from_party('/tmp/alice_train_labels_onehot.pth', src=ALICE, requires_grad=True)    
    y_bob_label_enc = crypten.load_from_party('/tmp/bob_train_labels_onehot.pth', src=BOB,requires_grad=True)    
    #crypten.print(x_alice_enc.size())
    #crypten.print(x_bob_enc.size())
    
    # Combine the feature sets: identical to Tutorial 3
    x_combined_enc = crypten.cat([x_alice_enc, x_bob_enc], dim=0)
    # Reshape to match the network architecture
    x_combined_enc = x_combined_enc.unsqueeze(1)
    
    y_combined_enc_one_hot = crypten.cat([y_alice_label_enc,y_bob_label_enc], dim=0)

    #crypten.print(f"Combined labels: {y_combined_enc_one_hot.get_plain_text()}")
    # Initialize a plaintext model and convert to CrypTen model
    pytorch_model = ExampleNet()
    model = crypten.nn.from_pytorch(pytorch_model, dummy_input)
    model.encrypt()
    # Set train mode
    model.train()
  
    # Define a loss function
    loss = crypten.nn.MSELoss()

    # Define training parameters
    learning_rate = 0.001
    num_epochs = 2
    batch_size = 1
    num_batches = x_combined_enc.size(0) // batch_size
    optimizer = crypten.optim.SGD(model.parameters(), learning_rate)
    
    rank = comm.get().get_rank()
    for i in range(num_epochs): 
        crypten.print(f"Epoch {i} in progress:")       
        
        for batch in range(num_batches):
            # define the start and end of the training mini-batch
            start, end = batch * batch_size, (batch + 1) * batch_size
                                    
            # construct CrypTensors out of training examples / labels
            x_train = x_combined_enc[start:end]
            y_train = y_combined_enc_one_hot[start:end]
            #crypten.print(f"y batch type: {type(y_batch)}", in_order=True)
            #crypten.print(f"y grad batch type : {type(y_batch.grad)}", in_order=True)
            #y_train = y_batch#.grad #crypten.cryptensor(y_batch, requires_grad=True)
            
            # perform forward pass:
            output = model(x_train)
            loss_value = loss(output, y_train)
            # set gradients to "zero" 
            model.zero_grad()

            # perform backward pass: 
            loss_value.backward()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value backward: {model.}", in_order=True)
            
            # update parameters
            optimizer.step()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tModel: {model}", in_order=True)

            
            # Print progress every batch:
            batch_loss = loss_value.get_plain_text()
            crypten.print(f"\tBatch {(batch + 1)} of {num_batches} Loss {batch_loss.item():.4f}")

run_encrypted_training()

Epoch 0 in progress:
	Batch 1 of 100 Loss 0.8801
	Batch 2 of 100 Loss 0.5371
	Batch 3 of 100 Loss 0.8192
	Batch 4 of 100 Loss 0.6561
	Batch 5 of 100 Loss 0.7095
	Batch 6 of 100 Loss 0.5411
	Batch 7 of 100 Loss 0.5493
	Batch 8 of 100 Loss 0.3901
	Batch 9 of 100 Loss 0.4320
	Batch 10 of 100 Loss 0.2897
	Batch 11 of 100 Loss 0.2656
	Batch 12 of 100 Loss 0.2381
	Batch 13 of 100 Loss 0.2815
	Batch 14 of 100 Loss 0.2949
	Batch 15 of 100 Loss 0.2564
	Batch 16 of 100 Loss 0.1531
	Batch 17 of 100 Loss 0.1482
	Batch 18 of 100 Loss 0.0440
	Batch 19 of 100 Loss 0.2299
	Batch 20 of 100 Loss 0.0801
	Batch 21 of 100 Loss 0.2201
	Batch 22 of 100 Loss 0.4733
	Batch 23 of 100 Loss 0.1030
	Batch 24 of 100 Loss 0.1026
	Batch 25 of 100 Loss 0.1107
	Batch 26 of 100 Loss 0.1172
	Batch 27 of 100 Loss 0.1911
	Batch 28 of 100 Loss 0.0943
	Batch 29 of 100 Loss 0.0647
	Batch 30 of 100 Loss 0.0417
	Batch 31 of 100 Loss 0.1451
	Batch 32 of 100 Loss 0.0301
	Batch 33 of 100 Loss 0.1169
	Batch 34 of 100 Loss 0.0095
	B

[None, None]

In [10]:
#Cleanup
import os

filenames = ['/tmp/alice_train.pth', 
             '/tmp/bob_train.pth', 
             '/tmp/alice_test.pth',
             '/tmp/bob_test.pth', 
             '/tmp/train_labels.pth',
             '/tmp/test_labels.pth']

for fn in filenames:
    if os.path.exists(fn): os.remove(fn)